# 年度恢复模型配对账户
使用相同2019预热后的OOF目标，双方2020现金起点、跨年账户连续。2020无模型明确禁用恢复；2021/22只选当年合格模型，原阈值及风险规则不变。6新原生账户，3个SOXX已存对照复用。

In [1]:
from pathlib import Path
import os,json,hashlib
import numpy as np,pandas as pd,qlib
from qlib.workflow import R
from src.workflow.minute_action_value import _source
from src.workflow.stage1_open_action_value import Stage1OpenActionValueSpec
from src.workflow.intraday_recovery_direct_allocation import IntradayRecoveryDirectAllocationSpec
from src.model.annual_recovery import AnnualRecoveryControllers
from src.backtest.full_path_tiered import replay_full_path
from src.backtest.stage1_ibkr import native_tiered_stage1
from src.backtest.qlib_continuous_position import QlibContinuousPosition
from src.analysis.integrated_allocation import paired_metrics,stationary_bootstrap_excess
from src.data.nas_market_data import archive_nas_recorder
root=Path('/Users/rziiiii/dev/workspace');card=root/'research/annual_recovery_account_20261015';p=json.loads((card/'plan.json').read_text());provider=root/'data/qlib/full_path_tiered_allocation_20261013';os.environ['MLFLOW_ALLOW_FILE_STORE']='true';qlib.init(provider_uri=str(provider),region='us')
source,sa=_source(root,p['model_source'],['inputs/plan','inputs/frame','inputs/selection','analysis/training_audit','models/2021','models/2022']);old,oa=_source(root,p['benchmark_source'],['inputs/plan','inputs/frame',*[f'analysis/native/soxx_{v}' for v in p['slippage_bps']]]);action,aa=_source(root,p['action_source'],['inputs/plan'])
for existing in [source,old]:
 for key in ['tariff','initial_cash','maximum_soxl_weight','minimum_cash_weight','maximum_drawdown']:assert existing['inputs/plan'][key]==p[key]
frame=source['inputs/frame'];frame=frame[frame.trade_date.between(p['evaluation_first_trade'],p['evaluation_last_trade'])].reset_index(drop=True);assert len(frame)==756
assert frame.trade_date.equals(old['inputs/frame'].trade_date.reset_index(drop=True))
for field in ['soxl_open','soxx_open','raw_soxl_open','raw_soxx_open','soxl_exit_open','soxx_exit_open']:np.testing.assert_allclose(frame[field],old['inputs/frame'][field].to_numpy(),rtol=0,atol=0)
available=pd.to_datetime(frame.recovery_feature_available_at,utc=True);decision=pd.to_datetime(frame.trade_date).dt.tz_localize('America/New_York')+pd.Timedelta(hours=10,minutes=31);assert available.le(decision).all()
recovery=IntradayRecoveryDirectAllocationSpec(root=root,**action['inputs/plan']['recovery_spec']);spec=Stage1OpenActionValueSpec(root=root,**action['inputs/plan']['action_spec']);selection=source['inputs/selection'];policy=AnnualRecoveryControllers({year:source[f'models/{year}'] for year in [2021,2022]},pd.DataFrame(source['analysis/training_audit']));baseline={'frozen_open_action_value_model':None,'frozen_recovery_model':None};books={};prefix=[]
with R.start(experiment_name=card.name,uri=str(card/'qlib_recorder')):
 rec=R.get_recorder();rec.save_objects(artifact_path='inputs',plan=p,source_audits={'models':sa,'benchmarks':oa,'action':aa},frame=frame,selection=selection,dated_policy=(root/'src/model/annual_recovery.py').read_text(),replay_code=(root/'src/backtest/full_path_tiered.py').read_text());rec.log_params(new_fits=0,maximum_new_native_cases=6)
for cost in p['slippage_bps']:
 for family in ['control','candidate']:
  kwargs={'controller':baseline} if family=='control' else {'controller_by_date':policy}
  replay=replay_full_path(frame,models=None,plan=p,slippage_bps=cost,recovery_spec=recovery,action_spec=spec,selection=selection,**kwargs)
  book=native_tiered_stage1(frame,replay,provider_uri=provider,tariff=p['tariff'],slippage_bps=cost,initial_cash=p['initial_cash'],freq='1min',opening_offset=pd.Timedelta(hours=9,minutes=47),position_type=QlibContinuousPosition);books[(family,cost)]=book;rec.save_objects(artifact_path='analysis/native',**{f'{family}_{cost}':book});rec.log_metrics(completed_new_native_cases=len([k for k in books if k[0]!='soxx']))
 control=books[('control',cost)]['authority'];candidate=books[('candidate',cost)]['authority'];mask=candidate.trade_date.dt.year.eq(2020);columns=['equity','ending_soxl_quantity','ending_soxx_quantity','ending_cash'];np.testing.assert_allclose(candidate.loc[mask,columns],control.loc[mask,columns],rtol=0,atol=1e-8);assert not candidate.loc[mask,'recovery_action'].any()
 prefix.append({'slippage_bps':cost,'2020_identical_sessions':int(mask.sum()),'2020_recovery_actions':int(candidate.loc[mask,'recovery_action'].sum())});books[('soxx',cost)]=old[f'analysis/native/soxx_{cost}']
rows=[];annual=[];uncertainty=[]
for (family,cost),book in books.items():
 path=book['authority'];bench=books[('soxx',cost)]['authority'];relative=np.log1p(path.strategy_return)-np.log1p(bench.strategy_return)
 rows.append({'family':family,'slippage_bps':cost,**paired_metrics(path,bench),'monitored_drawdown_pct':-100*path.monitored_drawdown.max(),'recovery_actions':int(path.recovery_action.sum()),'orders':len(book['orders']),'positive_126d_fraction':relative.rolling(126).sum().dropna().gt(0).mean()})
 for year in [2020,2021,2022]:
  mask=path.trade_date.dt.year.eq(year);annual.append({'family':family,'slippage_bps':cost,'year':year,'year_basis':'trade_year',**paired_metrics(path.loc[mask],bench.loc[mask]),'recovery_actions':int(path.loc[mask,'recovery_action'].sum())})
 if family=='candidate':
  for comparator in ['soxx','control']:uncertainty.append({'slippage_bps':cost,'comparator':comparator,**stationary_bootstrap_excess(path,books[(comparator,cost)]['authority'],mean_block=63,resamples=2000,alpha=.025,seed=20261015)})
table=pd.DataFrame(rows);annual=pd.DataFrame(annual);uncertainty=pd.DataFrame(uncertainty);cand=table[table.family.eq('candidate')].set_index('slippage_bps');ctl=table[table.family.eq('control')].set_index('slippage_bps');gates={'beat_control_all_costs':bool(cand.terminal_wealth.gt(ctl.terminal_wealth+1e-8).all()),'net_soxx_excess_all_costs':bool(cand.relative_wealth.gt(1).all()),'drawdown_30pct':bool(cand.max_drawdown_pct.ge(-30).all() and cand.monitored_drawdown_pct.ge(-30).all()),'cvar_no_worse':bool(cand.cvar5_pct.ge(cand.benchmark_cvar5_pct).all()),'rolling_70pct':bool(cand.positive_126d_fraction.ge(.7).all()),'annual_70pct':bool(annual[annual.family.eq('candidate')].groupby('slippage_bps').relative_wealth.apply(lambda x:x.gt(1).mean()).ge(.7).all()),'bootstrap_positive':bool(uncertainty.lower_bound_positive.all()),'all_year_recovery_models_available':False,'independent_validation':False}
policy_meta={'policy':'slim_card_recorder_first','workflow_config':'workflow.yaml','recorder_only_artifacts':['inputs/*','analysis/native/*']};summary={'experiment':card.name,'decision':'not_promoted_fixed_annual_recovery','gates':gates,'new_fits':0,'new_native_cases':6,'reused_benchmark_cases':3,'provider_uri':str(provider),'region':'us','as_of':p['as_of'],'recorder_id':rec.id,'experiment_id':str(rec.experiment_id),'recorder_uri':str(card/'qlib_recorder'),'recorder_local_dir':str(card/'qlib_recorder'/str(rec.experiment_id)/rec.id),'artifact_policy':policy_meta,'scope':p['scope'],'objective_achieved':False};rec.save_objects(artifact_path='analysis',comparison=table,annual=annual,uncertainty=uncertainty,prefix=pd.DataFrame(prefix),summary=summary);rec.log_metrics(registered_artifact_count=17)
for name,value in [('backtest_benchmark_comparison',table),('annual_metrics',annual),('uncertainty',uncertainty),('prefix_verification',pd.DataFrame(prefix))]:value.to_csv(card/(name+'.csv'),index=False)
import yaml
(card/'workflow.yaml').write_text(yaml.safe_dump({'experiment':card.name,'provider_uri':str(provider),'artifact_policy':policy_meta}));binding=json.loads((root/p['model_source']['card']/'nas_binding.json').read_text());binding.pop('evidence',None);(card/'nas_binding.json').write_text(json.dumps(binding,indent=2));a=Path(summary['recorder_local_dir'])/'artifacts';(card/'verification_manifest.json').write_text(json.dumps({'recorder_id':rec.id,'entries':[{'path':str(f.relative_to(card)),'sha256':hashlib.sha256(f.read_bytes()).hexdigest()} for f in sorted(a.rglob('*')) if f.is_file()]},indent=2));summary['nas_evidence']=archive_nas_recorder(card/'nas_binding.json',workspace_root=root,card=card,recorder_id=rec.id,experiment_id=str(rec.experiment_id));(card/'summary.json').write_text(json.dumps(summary,indent=2));print(table[['family','slippage_bps','return_pct','monitored_drawdown_pct','recovery_actions']].to_string(index=False))


# 年度恢复模型配对回测：未改善净收益或回撤

本轮0新拟合、6套新增Qlib原生账户，复用3套同窗同费SOXX账户。使用上轮已训练2021/2022恢复模型，2020因19条样本不达20门槛保持无模型。选择器只接收交易日期，按当年审核状态选择模型，并检查最后标签严格早于cutoff；不向前借用后年模型。

两组使用完全相同的2019年预热后OOF目标及修复后的首小时输入，均2020-01-02从10万美元现金起步，跨年持仓/净值/峰值/风险状态连续；最后退出2023-01-03，共756区间。开盘均执行原风险投影目标，候选只增加当年已合格的恢复分类器，原C=.3、概率阈值.825、恢复beta.6和alpha门.25不变。该比较不以之前未预热的参考作为唯一对照，避免把预热差异归给恢复模型。

| 滑点 | 无恢复对照净收益 | 年度恢复候选净收益 | SOXX净收益 | 候选监控最大回撤 | 恢复次数 |
|---|---:|---:|---:|---:|---:|
| 0bp | -5.10% | -5.68% | +42.06% | 32.95% | 6 |
| 2bp | -4.97% | -5.54% | +42.04% | 33.14% | 6 |
| 5bp | -6.09% | -7.21% | +42.01% | 33.25% | 7 |

2020三成本下均有253日候选/对照现金、股数、净值逐值一致，恢复次数为0，确认未回填未来模型。2021有模型但未达到冻结触发条件，也没有恢复。全部动作发生于2022：2bp该年候选-20.77%，对照-20.29%；全期三个成本均低于配对对照及SOXX，30%回撤、年度/滚动稳定与置信门失败。

这些结果说明该固定训练与执行配置没有贡献，不代表所有恢复信息都无效；动作稀少且2020缺模型的限制必须保留。不能通过现在降低.825阈值、调整C/动作beta、改变样本门或挑年份制造赢家。训练参考从2019开始，评价账户从2020开始，状态分布仍存在差异；本轮以完整自主账户而非局部标签准确率检验了这种迁移。它也不是原生产完整策略或未查看的独立前向证据。

决策not_promoted_fixed_annual_recovery。固定年度恢复路线关闭，不改生产。当前最大缺口仍是防守后恢复的可靠时间外信息，尤其2020缺少合格模型；任何后续机制需要先给出净优势信息或校准证据，不能继续只搜索触发参数。现金利息仍为原0口径，未凭缺失利率修改收益。稳定净SOXX超额目标未完成。

4项聚焦测试通过；原生现金/股数/费用/净值对账、2020配对前缀核对及NAS冷恢复逐对象哈希通过。共享回放新增可选日期控制器，原静态控制器路径保持原义，生产release未切换。
